In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("workspace.bronze.steam_games_raw")

# Garante padronização do nome da chave primária caso tenha ficado 'appid' ou 'app_id'
if "appid" in df_bronze.columns and "app_id" not in df_bronze.columns:
    df_bronze = df_bronze.withColumnRenamed("appid", "app_id")

total_linhas = df_bronze.count()
print(f"=== DIAGNÓSTICO DE QUALIDADE (CAMADA BRONZE) ===")
print(f"Total de registros brutos: {total_linhas}\n")

# 1. UNICIDADE: Verificar duplicatas de ID ou de Nome + Data
ids_distintos = df_bronze.select("app_id").distinct().count()
duplicados_id = total_linhas - ids_distintos
print(f"1. UNICIDADE -> IDs duplicados: {duplicados_id}")

# 2. COMPLETUDE: Contagem e percentual de valores nulos ou vazios nos atributos críticos
colunas_criticas = [
    "app_id", "name", "release_date", "price", 
    "supported_languages", "full_audio_languages", 
    "developers", "publishers", "genres", "categories"
]

metricas_nulos = []
for c in colunas_criticas:
    qtd_nulos = df_bronze.filter(
        F.col(c).isNull() | (F.trim(F.col(c)) == "") | (F.col(c) == "[]")
    ).count()
    pct = round((qtd_nulos / total_linhas) * 100, 2)
    metricas_nulos.append((c, qtd_nulos, pct))

df_completude = spark.createDataFrame(
    metricas_nulos, ["Atributo", "Qtd_Nulos_ou_Vazios", "Percentual_Incompleto_%"]
)
print("\n2. COMPLETUDE (Nulos, Vazios ou Listas Vazias '[]'):")
display(df_completude)

# 3. CONSISTÊNCIA: Verificar datas fora do padrão 'MMM dd, yyyy' (ex: 'Oct 2020' ou inválidas)
datas_padrao_completo = df_bronze.filter(
    F.expr("try_to_date(release_date, 'MMM d, yyyy')").isNotNull()
).count()
datas_inconsistentes = total_linhas - datas_padrao_completo
print(f"3. CONSISTÊNCIA -> Datas fora do formato padrão 'MMM d, yyyy': {datas_inconsistentes}")

# 4. ACURÁCIA E OUTLIERS: Preços negativos, preços extremos (> 500 USD) ou notas inválidas
df_num_check = (
    df_bronze
    .withColumn("preco_num", F.expr("try_cast(price as double)"))
    .withColumn("tempo_medio_num", F.expr("try_cast(average_playtime_forever as int)"))
)

precos_negativos = df_num_check.filter(F.col("preco_num") < 0).count()
outliers_preco = df_num_check.filter(F.col("preco_num") > 500).count()
outliers_playtime = df_num_check.filter(F.col("tempo_medio_num") > 500000).count()

print(f"4. ACURÁCIA -> Registros com preço negativo: {precos_negativos}")
print(f"5. OUTLIERS -> Jogos com preço > US$ 500 (pacotes/erros): {outliers_preco}")
print(f"5. OUTLIERS -> Jogos com tempo médio > 500.000 min (~95% de contas bots/idle): {outliers_playtime}")

=== DIAGNÓSTICO DE QUALIDADE (CAMADA BRONZE) ===
Total de registros brutos: 125855

1. UNICIDADE -> IDs duplicados: 0

2. COMPLETUDE (Nulos, Vazios ou Listas Vazias '[]'):


Atributo,Qtd_Nulos_ou_Vazios,Percentual_Incompleto_%
app_id,0,0.0
name,1,0.0
release_date,0,0.0
price,0,0.0
supported_languages,8462,6.72
full_audio_languages,8391,6.67
developers,0,0.0
publishers,8442,6.71
genres,8966,7.12
categories,8843,7.03


3. CONSISTÊNCIA -> Datas fora do formato padrão 'MMM d, yyyy': 0
4. ACURÁCIA -> Registros com preço negativo: 0
5. OUTLIERS -> Jogos com preço > US$ 500 (pacotes/erros): 3
5. OUTLIERS -> Jogos com tempo médio > 500.000 min (~95% de contas bots/idle): 0


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_base = spark.table("workspace.bronze.steam_games_raw")
if "appid" in df_silver_base.columns and "app_id" not in df_silver_base.columns:
    df_silver_base = df_silver_base.withColumnRenamed("appid", "app_id")

# Detecta automaticamente se o CSV do Kaggle sofreu o deslocamento de 'DiscountDLC count'
# Quando deslocado, a coluna 'supported_languages' recebe o texto de 'About the game' e 'full_audio_languages' recebe os idiomas de texto
amostra_deslocada = (
    df_silver_base
    .filter(F.col("full_audio_languages").contains("['English'"))
    .limit(10)
    .count()
) > 0

print(f"Correção de deslocamento de colunas ativada: {amostra_deslocada}")

if amostra_deslocada:
    col_dlc = F.col("about_the_game")
    col_idiomas_texto = F.col("full_audio_languages")
    col_idiomas_audio = F.col("reviews")
    col_windows = F.col("mac")
    col_mac = F.col("linux")
    col_linux = F.col("metacritic_score")
    col_positive = F.col("negative")
    col_negative = F.col("score_rank")
    col_avg_playtime = F.col("average_playtime_two_weeks")
    col_median_playtime = F.col("median_playtime_two_weeks")
    col_developers = F.col("publishers")
    col_publishers = F.col("categories")
    col_categories = F.col("genres")
    col_genres = F.col("tags")
else:
    col_dlc = F.col("discountdlc_count")
    col_idiomas_texto = F.col("supported_languages")
    col_idiomas_audio = F.col("full_audio_languages")
    col_windows = F.col("windows")
    col_mac = F.col("mac")
    col_linux = F.col("linux")
    col_positive = F.col("positive")
    col_negative = F.col("negative")
    col_avg_playtime = F.col("average_playtime_forever")
    col_median_playtime = F.col("median_playtime_forever")
    col_developers = F.col("developers")
    col_publishers = F.col("publishers")
    col_categories = F.col("categories")
    col_genres = F.col("genres")

# 1. Tipagem Segura (Casting) e Tratamento de Datas
df_tipado = (
    df_silver_base
    .withColumn("id_jogo", F.expr("try_cast(app_id as int)"))
    .withColumn("nome_jogo", F.trim(F.col("name")))
    .withColumn(
        "data_lancamento",
        F.coalesce(
            F.expr("try_to_date(release_date, 'MMM d, yyyy')"),
            F.expr("try_to_date(release_date, 'MMM dd, yyyy')"),
            F.expr("try_to_date(release_date, 'MMM yyyy')"),
            F.expr("try_to_date(release_date, 'yyyy-MM-dd')")
        )
    )
    .withColumn("preco_usd", F.coalesce(F.expr("try_cast(price as double)"), F.lit(0.0)))
    .withColumn("idade_minima", F.coalesce(F.expr("try_cast(required_age as int)"), F.lit(0)))
    .withColumn("qtd_dlc", F.coalesce(col_dlc.cast("int"), F.lit(0)))
    .withColumn("reviews_positivos", F.coalesce(col_positive.cast("int"), F.lit(0)))
    .withColumn("reviews_negativos", F.coalesce(col_negative.cast("int"), F.lit(0)))
    .withColumn("pico_jogadores_ccu", F.coalesce(F.expr("try_cast(peak_ccu as int)"), F.lit(0)))
    .withColumn("tempo_medio_jogo_min", F.coalesce(col_avg_playtime.cast("int"), F.lit(0)))
    .withColumn("mediana_tempo_jogo_min", F.coalesce(col_median_playtime.cast("int"), F.lit(0)))
    .withColumn("suporte_windows", F.coalesce(col_windows.cast("boolean"), F.lit(True)))
    .withColumn("suporte_mac", F.coalesce(col_mac.cast("boolean"), F.lit(False)))
    .withColumn("suporte_linux", F.coalesce(col_linux.cast("boolean"), F.lit(False)))
    .withColumn("_dev_raw", col_developers)
    .withColumn("_pub_raw", col_publishers)
    .withColumn("_cat_raw", col_categories)
    .withColumn("_gen_raw", col_genres)
    .withColumn("_lang_txt_raw", col_idiomas_texto)
    .withColumn("_lang_aud_raw", col_idiomas_audio)
)

# 2. Deduplicação e Limpeza
janela_dedup = Window.partitionBy("id_jogo").orderBy(F.col("_data_ingestao").desc())

df_limpo = (
    df_tipado
    .filter(F.col("id_jogo").isNotNull() & F.col("nome_jogo").isNotNull() & (F.col("nome_jogo") != ""))
    .filter(F.col("data_lancamento").isNotNull())
    .withColumn("rn", F.row_number().over(janela_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .filter((F.col("preco_usd") >= 0.0) & (F.col("preco_usd") <= 500.0))
    .withColumn(
        "desenvolvedora",
        F.when(F.col("_dev_raw").isNull() | (F.trim(F.col("_dev_raw")) == ""), "Não Informado")
         .otherwise(F.regexp_replace(F.col("_dev_raw"), r"[\[\]'\"\\]", ""))
    )
    .withColumn(
        "publicadora",
        F.when(F.col("_pub_raw").isNull() | (F.trim(F.col("_pub_raw")) == ""), "Não Informado")
         .otherwise(F.regexp_replace(F.col("_pub_raw"), r"[\[\]'\"\\]", ""))
    )
    .withColumn(
        "faixa_proprietarios_estimada",
        F.when(F.col("estimated_owners").isNull() | (F.trim(F.col("estimated_owners")) == ""), "0 - 20000")
         .otherwise(F.trim(F.col("estimated_owners")))
    )
)

# 3. Extração das Flags de Português-BR, Gêneros e Modalidades
df_silver_final = (
    df_limpo
    .withColumn("idiomas_texto_limpo", F.regexp_replace(F.coalesce(F.col("_lang_txt_raw"), F.lit("")), r"[\[\]'\"\\]", ""))
    .withColumn("idiomas_audio_limpo", F.regexp_replace(F.coalesce(F.col("_lang_aud_raw"), F.lit("")), r"[\[\]'\"\\]", ""))
    .withColumn("generos_limpos", F.regexp_replace(F.coalesce(F.col("_gen_raw"), F.lit("")), r"[\[\]'\"\\]", ""))
    .withColumn("categorias_limpas", F.regexp_replace(F.coalesce(F.col("_cat_raw"), F.lit("")), r"[\[\]'\"\\]", ""))
    
    # Flags de Português-BR
    .withColumn("tem_texto_ptbr", F.lower(F.col("idiomas_texto_limpo")).rlike("(portuguese - brazil|portuguese-brazil|brazil|portuguese)"))
    .withColumn("tem_audio_ptbr", F.lower(F.col("idiomas_audio_limpo")).rlike("(portuguese - brazil|portuguese-brazil|brazil|portuguese)"))
    .withColumn(
        "nivel_localizacao_br",
        F.when(F.col("tem_audio_ptbr") == True, "Dublagem Completa (Áudio + Texto)")
         .when(F.col("tem_texto_ptbr") == True, "Apenas Texto / Legenda PT-BR")
         .otherwise("Sem Suporte ao Português-BR")
    )
    .withColumn(
        "qtd_idiomas_suportados",
        F.when(F.trim(F.col("idiomas_texto_limpo")) == "", 1)
         .otherwise(F.size(F.split(F.col("idiomas_texto_limpo"), ",")))
    )
    .withColumn(
        "genero_principal",
        F.when(F.trim(F.col("generos_limpos")) == "", "Não Classificado")
         .otherwise(F.trim(F.split(F.col("generos_limpos"), ",").getItem(0)))
    )
    .withColumn("is_indie", F.lower(F.col("generos_limpos")).contains("indie"))
    .withColumn(
        "modalidade_jogador",
        F.when(
            F.lower(F.col("categorias_limpas")).rlike("(multi-player|multiplayer|co-op|coop|pvp|mmo)"),
            "Multiplayer / Co-op"
        ).when(
            F.lower(F.col("categorias_limpas")).rlike("(single-player|singleplayer)"),
            "Exclusivo Single-player"
        ).otherwise("Outros / Não Especificado")
    )
    .withColumn("is_free_to_play", F.col("preco_usd") == 0.0)
    .withColumn(
        "faixa_preco",
        F.when(F.col("preco_usd") == 0.0, "1. Gratuito (Free-to-Play)")
         .when(F.col("preco_usd") <= 10.0, "2. Até US$ 10 (Econômico)")
         .when(F.col("preco_usd") <= 30.0, "3. US$ 10.01 a US$ 30 (Intermediário)")
         .otherwise("4. Acima de US$ 30 (Premium / AAA)")
    )
    .withColumn("total_reviews", F.col("reviews_positivos") + F.col("reviews_negativos"))
    .withColumn(
        "taxa_aprovacao_pct",
        F.when(F.col("total_reviews") > 0, F.round((F.col("reviews_positivos") / F.col("total_reviews")) * 100, 2))
         .otherwise(None)
    )
    .withColumn("_data_processamento_silver", F.current_timestamp())
)

colunas_silver = [
    "id_jogo", "nome_jogo", "data_lancamento", "preco_usd", "is_free_to_play", "faixa_preco",
    "idade_minima", "qtd_dlc", "desenvolvedora", "publicadora", "genero_principal", "generos_limpos",
    "is_indie", "modalidade_jogador", "categorias_limpas", "tem_texto_ptbr", "tem_audio_ptbr",
    "nivel_localizacao_br", "qtd_idiomas_suportados", "suporte_windows", "suporte_mac", "suporte_linux",
    "reviews_positivos", "reviews_negativos", "total_reviews", "taxa_aprovacao_pct",
    "pico_jogadores_ccu", "tempo_medio_jogo_min", "mediana_tempo_jogo_min",
    "faixa_proprietarios_estimada", "_data_ingestao", "_data_processamento_silver"
]

df_silver_salvar = df_silver_final.select(*colunas_silver)

(
    df_silver_salvar.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.steam_games_clean")
)

# Validação imediata: mostra quantos jogos ficaram em cada nível de localização e modalidade
display(df_silver_salvar.groupBy("nivel_localizacao_br", "modalidade_jogador").count().orderBy(F.col("count").desc()))

Correção de deslocamento de colunas ativada: True


nivel_localizacao_br,modalidade_jogador,count
Sem Suporte ao Português-BR,Exclusivo Single-player,77721
Sem Suporte ao Português-BR,Multiplayer / Co-op,15808
Apenas Texto / Legenda PT-BR,Exclusivo Single-player,13201
Sem Suporte ao Português-BR,Outros / Não Especificado,9849
Apenas Texto / Legenda PT-BR,Multiplayer / Co-op,4019
Dublagem Completa (Áudio + Texto),Exclusivo Single-player,3923
Dublagem Completa (Áudio + Texto),Multiplayer / Co-op,1054
Apenas Texto / Legenda PT-BR,Outros / Não Especificado,234
Dublagem Completa (Áudio + Texto),Outros / Não Especificado,42


In [0]:
%sql
COMMENT ON TABLE workspace.silver.steam_games_clean IS 
'Tabela Silver com dados de jogos da Steam limpos, tipados, deduplicados, sem outliers de preço e enriquecidos com flags de localização para Português-BR.';

-- Validação rápida: Quantos jogos têm suporte ao Português-BR na base limpa?
SELECT 
    nivel_localizacao_br,
    COUNT(*) AS qtd_jogos,
    ROUND(AVG(preco_usd), 2) AS preco_medio_usd,
    ROUND(AVG(taxa_aprovacao_pct), 2) AS media_aprovacao_pct
FROM workspace.silver.steam_games_clean
GROUP BY nivel_localizacao_br
ORDER BY qtd_jogos DESC;

nivel_localizacao_br,qtd_jogos,preco_medio_usd,media_aprovacao_pct
Sem Suporte ao Português-BR,107498,4.36,0.03
Dublagem Completa (Áudio + Texto),18353,7.31,0.05


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_base = spark.table("workspace.bronze.steam_games_raw")
if "appid" in df_silver_base.columns and "app_id" not in df_silver_base.columns:
    df_silver_base = df_silver_base.withColumnRenamed("appid", "app_id")

# 1. Consolidação segura (protege contra deslocamento de 1 coluna no CSV do Kaggle)
col_idiomas_texto = F.concat_ws(" ", F.col("supported_languages"), F.col("full_audio_languages"))
col_idiomas_audio = F.concat_ws(" ", F.col("full_audio_languages"), F.col("reviews"))
col_generos_bruto = F.coalesce(
    F.when(F.col("genres").rlike("(?i)(Action|Indie|Adventure|RPG|Strategy|Simulation|Casual|Sports|Racing|Free)"), F.col("genres")),
    F.when(F.col("categories").rlike("(?i)(Action|Indie|Adventure|RPG|Strategy|Simulation|Casual|Sports|Racing)"), F.col("categories")),
    F.when(F.col("tags").rlike("(?i)(Action|Indie|Adventure|RPG|Strategy|Simulation|Casual|Sports|Racing)"), F.col("tags")),
    F.col("genres")
)
col_categorias_bruto = F.concat_ws(" ", F.col("categories"), F.col("genres"), F.col("publishers"))

# 2. Tipagem Segura (Casting) e Tratamento de Datas
df_tipado = (
    df_silver_base
    .withColumn("id_jogo", F.expr("try_cast(app_id as int)"))
    .withColumn("nome_jogo", F.trim(F.col("name")))
    .withColumn(
        "data_lancamento",
        F.coalesce(
            F.expr("try_to_date(release_date, 'MMM d, yyyy')"),
            F.expr("try_to_date(release_date, 'MMM dd, yyyy')"),
            F.expr("try_to_date(release_date, 'MMM yyyy')"),
            F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
            F.to_date(F.lit("2020-01-01"))
        )
    )
    .withColumn(
        "preco_usd", 
        F.coalesce(
            F.expr("try_cast(price as double)"), 
            F.expr("try_cast(required_age as double)"), 
            F.lit(0.0)
        )
    )
    .withColumn("idade_minima", F.coalesce(F.expr("try_cast(required_age as int)"), F.lit(0)))
    .withColumn("qtd_dlc", F.coalesce(F.expr("try_cast(discountdlc_count as int)"), F.lit(0)))
    .withColumn(
        "reviews_positivos", 
        F.coalesce(
            F.expr("try_cast(positive as int)"), 
            F.expr("try_cast(user_score as int)"), 
            F.lit(0)
        )
    )
    .withColumn(
        "reviews_negativos", 
        F.coalesce(
            F.expr("try_cast(negative as int)"), 
            F.expr("try_cast(score_rank as int)"), 
            F.lit(0)
        )
    )
    .withColumn("pico_jogadores_ccu", F.coalesce(F.expr("try_cast(peak_ccu as int)"), F.lit(0)))
    .withColumn(
        "tempo_medio_jogo_min", 
        F.coalesce(
            F.expr("try_cast(average_playtime_forever as int)"), 
            F.expr("try_cast(average_playtime_two_weeks as int)"), 
            F.lit(0)
        )
    )
    .withColumn(
        "mediana_tempo_jogo_min", 
        F.coalesce(
            F.expr("try_cast(median_playtime_forever as int)"), 
            F.expr("try_cast(median_playtime_two_weeks as int)"), 
            F.lit(0)
        )
    )
    .withColumn("suporte_windows", F.coalesce(F.expr("try_cast(windows as boolean)"), F.lit(True)))
    .withColumn("suporte_mac", F.coalesce(F.expr("try_cast(mac as boolean)"), F.lit(False)))
    .withColumn("suporte_linux", F.coalesce(F.expr("try_cast(linux as boolean)"), F.lit(False)))
)

# 3. Deduplicação e Limpeza
janela_dedup = Window.partitionBy("id_jogo").orderBy(F.col("_data_ingestao").desc())

df_limpo = (
    df_tipado
    .filter(F.col("id_jogo").isNotNull() & F.col("nome_jogo").isNotNull() & (F.col("nome_jogo") != ""))
    .withColumn("rn", F.row_number().over(janela_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .filter((F.col("preco_usd") >= 0.0) & (F.col("preco_usd") <= 500.0))
    .withColumn(
        "desenvolvedora",
        F.when(F.col("developers").isNull() | (F.trim(F.col("developers")) == ""), "Não Informado")
         .otherwise(F.regexp_replace(F.col("developers"), r"[\[\]'\"\\]", ""))
    )
    .withColumn(
        "publicadora",
        F.when(F.col("publishers").isNull() | (F.trim(F.col("publishers")) == ""), "Não Informado")
         .otherwise(F.regexp_replace(F.col("publishers"), r"[\[\]'\"\\]", ""))
    )
    .withColumn(
        "faixa_proprietarios_estimada",
        F.when(F.col("estimated_owners").isNull() | (F.trim(F.col("estimated_owners")) == ""), "0 - 20000")
         .otherwise(F.trim(F.col("estimated_owners")))
    )
)

# 4. Extração Flexível de Português-BR, Gêneros e Modalidades
df_silver_final = (
    df_limpo
    .withColumn("idiomas_texto_limpo", F.regexp_replace(col_idiomas_texto, r"[\[\]'\"\\]", ""))
    .withColumn("idiomas_audio_limpo", F.regexp_replace(col_idiomas_audio, r"[\[\]'\"\\]", ""))
    .withColumn("generos_limpos", F.regexp_replace(F.coalesce(col_generos_bruto, F.lit("Action")), r"[\[\]'\"\\]", ""))
    .withColumn("categorias_limpas", F.regexp_replace(col_categorias_bruto, r"[\[\]'\"\\]", ""))
    
    # Captura qualquer variação de Português / Brasil na coluna de idiomas
    .withColumn("tem_texto_ptbr", F.lower(F.col("idiomas_texto_limpo")).rlike("(portuguese|brazil|português|pt-br)"))
    .withColumn("tem_audio_ptbr", F.lower(F.col("idiomas_audio_limpo")).rlike("(portuguese|brazil|português|pt-br)"))
    .withColumn(
        "nivel_localizacao_br",
        F.when(F.col("tem_audio_ptbr") == True, "Dublagem Completa (Áudio + Texto)")
         .when(F.col("tem_texto_ptbr") == True, "Apenas Texto / Legenda PT-BR")
         .otherwise("Sem Suporte ao Português-BR")
    )
    .withColumn(
        "qtd_idiomas_suportados",
        F.when(F.trim(F.col("idiomas_texto_limpo")) == "", 1)
         .otherwise(F.size(F.split(F.col("idiomas_texto_limpo"), ",")))
    )
    .withColumn(
        "genero_principal",
        F.when(F.trim(F.col("generos_limpos")) == "", "Não Classificado")
         .otherwise(F.trim(F.split(F.col("generos_limpos"), ",").getItem(0)))
    )
    .withColumn("is_indie", F.lower(F.col("generos_limpos")).contains("indie"))
    .withColumn(
        "modalidade_jogador",
        F.when(
            F.lower(F.col("categorias_limpas")).rlike("(multi-player|multiplayer|co-op|coop|online|pvp|mmo)"),
            "Multiplayer / Co-op"
        ).when(
            F.lower(F.col("categorias_limpas")).rlike("(single-player|singleplayer)"),
            "Exclusivo Single-player"
        ).otherwise("Exclusivo Single-player")
    )
    .withColumn("is_free_to_play", F.col("preco_usd") == 0.0)
    .withColumn(
        "faixa_preco",
        F.when(F.col("preco_usd") == 0.0, "1. Gratuito (Free-to-Play)")
         .when(F.col("preco_usd") <= 10.0, "2. Até US$ 10 (Econômico)")
         .when(F.col("preco_usd") <= 30.0, "3. US$ 10.01 a US$ 30 (Intermediário)")
         .otherwise("4. Acima de US$ 30 (Premium / AAA)")
    )
    .withColumn("total_reviews", F.col("reviews_positivos") + F.col("reviews_negativos"))
    .withColumn(
        "taxa_aprovacao_pct",
        F.when(F.col("total_reviews") > 0, F.round((F.col("reviews_positivos") / F.col("total_reviews")) * 100, 2))
         .otherwise(None)
    )
    .withColumn("_data_processamento_silver", F.current_timestamp())
)

colunas_silver = [
    "id_jogo", "nome_jogo", "data_lancamento", "preco_usd", "is_free_to_play", "faixa_preco",
    "idade_minima", "qtd_dlc", "desenvolvedora", "publicadora", "genero_principal", "generos_limpos",
    "is_indie", "modalidade_jogador", "categorias_limpas", "tem_texto_ptbr", "tem_audio_ptbr",
    "nivel_localizacao_br", "qtd_idiomas_suportados", "suporte_windows", "suporte_mac", "suporte_linux",
    "reviews_positivos", "reviews_negativos", "total_reviews", "taxa_aprovacao_pct",
    "pico_jogadores_ccu", "tempo_medio_jogo_min", "mediana_tempo_jogo_min",
    "faixa_proprietarios_estimada", "_data_ingestao", "_data_processamento_silver"
]

df_silver_salvar = df_silver_final.select(*colunas_silver)

(
    df_silver_salvar.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.steam_games_clean")
)

print("Distribuição de Localização PT-BR após atualização da Silver:")
display(df_silver_salvar.groupBy("nivel_localizacao_br").count())

Distribuição de Localização PT-BR após atualização da Silver:


nivel_localizacao_br,count
Apenas Texto / Legenda PT-BR,120
Sem Suporte ao Português-BR,103258
Dublagem Completa (Áudio + Texto),22473
